# Enedis — entrenament Transformer v3.1 (Kaggle)

Un sol procés de 80 èpoques. Després de **cada** època es guarda `latest.pt`.
Si Kaggle s'atura, una sessió nova amb `RESUME=True` continua a l'època següent.

Prerequisit: el dataset del repositori (codi + `data/`) muntat a `/kaggle/input/...` o el cwd del notebook.

In [ ]:
# ── Configuració (edita només això) ──────────────────────────────────────────
RESUME = False

# Si RESUME=True:
#   None → /kaggle/working/DataChallenge_checkpoints/latest.pt
#   o una ruta explícita al checkpoint complet
RESUME_CHECKPOINT = None

KAGGLE_CKPT_DIR = "/kaggle/working/DataChallenge_checkpoints"

In [ ]:
import os
import sys
from pathlib import Path

# Arrel del repo: cwd, o el dataset de Kaggle si el notebook no està al repo
ROOT = Path.cwd()
if not (ROOT / "lstm" / "main_train_combined.py").exists():
    candidates = list(Path("/kaggle/input").glob("*/lstm/main_train_combined.py"))
    if candidates:
        ROOT = candidates[0].parents[1]
        os.chdir(ROOT)
        print(f"cwd → {ROOT}")
    else:
        raise FileNotFoundError(
            "No s'ha trobat lstm/main_train_combined.py. Afegeix el repo com a dataset.")

sys.path.insert(0, str(ROOT))
print("ROOT:", ROOT)

In [ ]:
import lstm.main_train_combined as train_script

train_script.RESUME = RESUME
train_script.RESUME_FROM = RESUME_CHECKPOINT

print("RESUME         :", train_script.RESUME)
print("RESUME_FROM    :", train_script.RESUME_FROM)
print("ckpt_dir       :", train_script.resolve_ckpt_dir())

In [ ]:
# Entrenament únic (80 èpoques, checkpoint després de cada època)
train_script.main()

In [ ]:
from pathlib import Path
import json

ckpt_dir = Path("/kaggle/working/DataChallenge_checkpoints")
if not ckpt_dir.exists():
    ckpt_dir = train_script.resolve_ckpt_dir()

print("Checkpoints a", ckpt_dir)
if ckpt_dir.exists():
    for p in sorted(ckpt_dir.iterdir()):
        print(f"  {p.name:24s}  {p.stat().st_size/1e6:.2f} MB")
else:
    print("  (directori inexistent)")

In [ ]:
from pathlib import Path
import json

ckpt_dir = Path("/kaggle/working/DataChallenge_checkpoints")
if not ckpt_dir.exists():
    ckpt_dir = train_script.resolve_ckpt_dir()

status_path = ckpt_dir / "training_status.json"
status = {}
if status_path.exists():
    status = json.loads(status_path.read_text(encoding="utf-8"))

latest = ckpt_dir / "latest.pt"
best_full = ckpt_dir / "best.pt"
best_weights = Path(status.get("best_weights", train_script.COMBINED_CKPT))

print("Training finished")
print(f"Last checkpoint: {latest if latest.exists() else status.get('latest')}")
print(f"Best checkpoint/model: {best_weights}")
print(f"Last completed epoch: {status.get('last_completed_epoch', '?')}")
if best_full.exists():
    print(f"Best full checkpoint (resume): {best_full}")